# Pivot-English robustness

For each (model, prompt-group) cell, compare the favored side under direct within-language judging vs. under translation-mediated English judging. Reports the number of cells whose favored side changes.


In [ ]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd

from src.conflicts import get_conflict
from src.statistics.aggregation import load_run_scores


## Config


In [ ]:
CONFLICT = 'ru_ua'
DIRECT_ROOT = '../../results'
PIVOT_ROOT = '../../results_pivot_english'
JUDGE_LABEL = 'llama3-8b'
JUDGE_PROMPT = 'primary'

MODELS = [
    'mistral-7b-instruct-v0.3', 'qwen25-7b', 'llama3-8b',
    'gemini-3-pro-preview', 'deepseek-7b-chat', 'falcon-7b-instruct',
    'moonshot-v1-8k', 'gpt-5.1',
]


## Prompt groups


In [ ]:
PROMPT_GROUPS = {
    'baseline': ['neutral_1', 'neutral_2', 'neutral_3'],
    'social_influence': ['influencer_1', 'influencer_2', 'twitter_1', 'twitter_2', 'social_media_reporter_1', 'social_media_reporter_2'],
    'news': ['telegram_channel_1', 'telegram_channel_2', 'news_article_1', 'news_article_2'],
    'journalist_voice': ['social_media_reporter_1', 'social_media_reporter_2', 'telegram_channel_1', 'news_article_1', 'news_article_2'],
    'country_aligned': ['telegram_channel_1', 'telegram_channel_2', 'twitter_1', 'twitter_2', 'influencer_1', 'influencer_2', 'news_article_1'],
}


## Compare favored side per cell


In [ ]:
conflict = get_conflict(CONFLICT)
side_a, side_b = conflict.sides

def cell_mean(model, prompts, side, root):
    scores = []
    for p in prompts:
        s = load_run_scores(results_root=root, conflict=conflict, judge_label=JUDGE_LABEL, judge_prompt=JUDGE_PROMPT, translator_label=model, prompt_name=p, side=side)
        scores.extend(s.values())
    return float(np.mean(scores)) if scores else None

def favored_side(model, prompts, root):
    a = cell_mean(model, prompts, side_a, root)
    b = cell_mean(model, prompts, side_b, root)
    if a is None or b is None:
        return None
    return side_a.code if a < b else side_b.code if b < a else 'tie'

rows = []
flipped = 0
total = 0
for model in MODELS:
    for group, templates in PROMPT_GROUPS.items():
        direct = favored_side(model, templates, DIRECT_ROOT)
        pivot = favored_side(model, templates, PIVOT_ROOT)
        flip = direct is not None and pivot is not None and direct != pivot
        if direct is not None and pivot is not None:
            total += 1
            if flip:
                flipped += 1
        rows.append({'model': model, 'group': group, 'direct': direct, 'pivot_english': pivot, 'flipped': flip})

df = pd.DataFrame(rows)
print(f'{flipped} / {total} cells changed favored side under pivot-English judging')
df
